In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.4 The Central-Force Problem and Orbits

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume II — Analytical Mechanics",
    number="2.4",
    title="The Central-Force Problem and Orbits",
    blurb="Reducing two bodies to one radial coordinate: the effective "
    "potential, orbits as motion in 1-D, the Binet equation, and why "
    "only the inverse-square (and Hooke) law gives closed orbits.",
    difficulty="intermediate",
    estimate="75–100 min",
)

## Notebook overview

In [§1.4](../01-elementary-mechanics/kepler-orbits.ipynb) we *integrated* Kepler orbits directly in Cartesian coordinates
and watched the ellipse close. This notebook is the **analytical complement**:
instead of pushing $(x,y)$ forward in time, we use the two conserved quantities
(energy and angular momentum) to collapse the whole two-dimensional problem onto a
single radial coordinate, and then *solve for the shape of the orbit itself*.

The engine is one idea. Conservation of angular momentum (the rotational
symmetry of [§2.2](noether.ipynb); the cyclic angle of [§2.3](hamiltonian-phase-flow.ipynb)) lets us absorb the angular
motion into an **effective potential** $V_{\mathrm{eff}}(r)$, after which the
radial coordinate moves *exactly like a one-dimensional particle* in that
potential: the 1-D Hamiltonian motion of [§2.3](hamiltonian-phase-flow.ipynb). Reading
$V_{\mathrm{eff}}(r)$ like a 1-D energy diagram tells us at a glance which orbits
are circular (sit at the minimum), which are bound (oscillate between two turning
points), and which escape. To get the *geometric* shape $r(\varphi)$ we change
variables to $u=1/r$ and obtain the **Binet equation**, a linear oscillator
equation whose solution is a conic section: the ellipse of [§1.4](../01-elementary-mechanics/kepler-orbits.ipynb), now derived
rather than discovered.

We build a small toolkit (`V_eff`, its derivatives, a turning-point finder, the
Binet right-hand side, and a 2-D Cartesian integrator for cross-checks) and put
it to work: locating circular orbits and testing their stability, finding
perihelion and aphelion as turning points, integrating the Binet equation and
matching it to the analytic conic, animating the closed inverse-square ellipse,
and finally animating the **precessing rosette** that appears the moment the force
law departs from $1/r^2$: the analytical sibling of the precession experiment
of [§1.4](../01-elementary-mechanics/kepler-orbits.ipynb).

> **How to read the checks.** Each exercise ends with a validation that compares
> a computed result to an expected physical fact. A ✗ does *not* by itself mean
> the physics is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a sign,
> a unit, an array order), or simply too tight a tolerance. Treat a ✗ as a prompt
> to locate the discrepancy; passing is strong evidence of correctness, not
> proof.

> **Scope.** This is a working review, not a textbook chapter. For the full
> derivation of the central-force reduction, the orbit equation, and Bertrand's
> theorem, see Nolting, *Theoretical Physics 2* {cite}`nolting2`, Goldstein,
> Poole & Safko {cite}`goldstein` (ch. 3), and Landau & Lifshitz, *Mechanics*
> {cite}`landau_mechanics` (§III).

## Theory in brief

### Reduction to one body and one radial coordinate

A two-body system interacting through a potential that depends only on their
separation reduces to a **single** fictitious particle of reduced mass $\mu =
m_1 m_2/(m_1+m_2)$ moving in the central potential $V(r)$ about a fixed centre
([§1.4](../01-elementary-mechanics/kepler-orbits.ipynb)). Because the force is central, the angular momentum

```{math}
:label: eq-L
\mathbf L = \mu\,\mathbf r \times \dot{\mathbf r}, \qquad
L = \mu r^2 \dot\varphi = \text{const},
```

is conserved: this is the conserved charge of rotational symmetry ([§2.2](noether.ipynb))
and of the cyclic angle $\varphi$ ([§2.3](hamiltonian-phase-flow.ipynb)). Its constancy confines the
motion to a plane and fixes $\dot\varphi = L/(\mu r^2)$, so the angular motion is
no longer an independent unknown.

### The effective potential: orbits as 1-D motion

Substituting $\dot\varphi = L/(\mu r^2)$ into the energy $E = \tfrac12\mu(\dot
r^2 + r^2\dot\varphi^2) + V(r)$ removes $\varphi$ entirely and leaves a purely
radial energy equation,

```{math}
:label: eq-veff
E = \tfrac12\mu\dot r^2 + V_{\mathrm{eff}}(r), \qquad
V_{\mathrm{eff}}(r) = V(r) + \frac{L^2}{2\mu r^2}.
```

This is the conceptual heart of the chapter: **the radial coordinate moves
exactly like a one-dimensional particle of mass $\mu$ in the potential
$V_{\mathrm{eff}}$** (a 1-D Hamiltonian system, [§2.3](hamiltonian-phase-flow.ipynb)). The extra term
$L^2/(2\mu r^2)$ is the **centrifugal barrier** (the angular-momentum cost of
approaching the centre), and it is what keeps a particle with $L\neq 0$ from
falling in.

### Circular orbits, turning points, and stability

Reading {eq}`eq-veff` as a 1-D energy diagram classifies every orbit:

- A **circular** orbit sits at a minimum of $V_{\mathrm{eff}}$, where the inward
  force balances the centrifugal barrier. For the attractive inverse-square
  potential $V=-GM\mu/r$ this gives

```{math}
:label: eq-rc
\left.\frac{dV_{\mathrm{eff}}}{dr}\right|_{r_c}=0
\;\Longrightarrow\;
r_c = \frac{L^2}{GM\mu^2},
```

  and the orbit is **stable** when $V_{\mathrm{eff}}''(r_c)>0$.
- A **bound** orbit ($E<0$) oscillates radially between two **turning points**,
  the roots of

```{math}
:label: eq-turning
E = V_{\mathrm{eff}}(r) \qquad (\dot r = 0),
```

  which are the **perihelion** $r_{\min}$ and **aphelion** $r_{\max}$.
- An **unbound** orbit ($E\geq 0$) has a single turning point and escapes: the
  scattering states that open [§2.5](scattering.ipynb).

### The Binet equation and the conic-section orbit

To get the *shape* $r(\varphi)$ rather than the timing, substitute $u=1/r$ and use
{eq}`eq-L` to trade $d/dt$ for $d/d\varphi$. The radial equation becomes the
**Binet equation** $d^2u/d\varphi^2 + u = -\,\mu/(L^2u^2)\,F(1/u)$, where
$F=-dV/dr$. For the inverse-square force the right-hand side is constant,

```{math}
:label: eq-binet
\frac{d^2u}{d\varphi^2} + u = \frac{GM\mu^2}{L^2},
```

a *linear oscillator* in $\varphi$, and its solution is a conic section,

```{math}
:label: eq-conic
u(\varphi)=\frac{GM\mu^2}{L^2}\bigl(1+e\cos\varphi\bigr), \qquad
e=\sqrt{1+\frac{2EL^2}{G^2M^2\mu^3}}.
```

The eccentricity $e$ sorts the orbits: $e=0$ circle, $0<e<1$ ellipse, $e=1$
parabola, $e>1$ hyperbola. Because the oscillator in {eq}`eq-binet` has unit
angular frequency in $\varphi$, the perihelion-to-aphelion **apsidal angle** is
exactly $\pi$: the orbit closes after one revolution. **Bertrand's theorem** says
only two force laws share this property: the inverse-square ($\propto 1/r^2$) and
the Hooke ($\propto r$) law; every other central force precesses ([§1.4](../01-elementary-mechanics/kepler-orbits.ipynb)).

---
## Setup

We work in **reduced units** $GM = 1$ and $\mu = 1$, so the circular radius
{eq}`eq-rc` is simply $r_c = L^2$. The helpers below are the reusable core:

- `V_eff(r, L)`: the effective potential {eq}`eq-veff`, with `dVeff_dr` and
  `d2Veff_dr2` for locating and classifying circular orbits.
- `turning_points(E, L)`: perihelion/aphelion as roots of {eq}`eq-turning`,
  found by **bracketing each sign change** of $E-V_{\mathrm{eff}}(r)$ on a scan
  and refining with `brentq` (no hard-coded brackets, a turning point can sit
  just inside a guessed one).
- `binet_rhs(phi, y, L)`: the Binet right-hand side {eq}`eq-binet` for the
  inverse-square law, plus a perturbed variant in Exercise 6.
- `cartesian_rhs(t, s)`: the 2-D Newtonian integrator (reused from [§1.4](../01-elementary-mechanics/kepler-orbits.ipynb)) for an
  independent cross-check.

The geometry (a particle located by polar coordinates $(r,\varphi)$ about a fixed
force centre) is sketched in {numref}`fig-central-geometry`.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import draw, mechanics, validate
from ecp.animate import show

GM = 1.0  # reduced units: GM = μ = 1 throughout
MU = 1.0


def V(r):
    """Attractive inverse-square potential energy V(r) = −GM·μ/r.

    Parameters
    ----------
    r : float or numpy.ndarray
        Radius.

    Returns
    -------
    float or numpy.ndarray
        The potential energy.
    """
    return -GM * MU / r


def V_eff(r, L):
    """Effective radial potential V_eff = V(r) + L^2/(2μ r^2) (eq-veff).

    The centrifugal barrier reduces the orbit to 1-D radial motion.

    Parameters
    ----------
    r : float or numpy.ndarray
        Radius.
    L : float
        Angular momentum.

    Returns
    -------
    float or numpy.ndarray
        The effective potential.
    """
    return V(r) + L**2 / (2.0 * MU * r**2)


def dVeff_dr(r, L):
    """First derivative dV_eff/dr (zero at a circular orbit, eq-rc).

    Parameters
    ----------
    r : float or numpy.ndarray
        Radius.
    L : float
        Angular momentum.

    Returns
    -------
    float or numpy.ndarray
        The derivative.
    """
    return GM * MU / r**2 - L**2 / (MU * r**3)


def d2Veff_dr2(r, L):
    """Second derivative d^2V_eff/dr^2 (>0 means the circular orbit is stable).

    Parameters
    ----------
    r : float or numpy.ndarray
        Radius.
    L : float
        Angular momentum.

    Returns
    -------
    float or numpy.ndarray
        The second derivative.
    """
    return -2.0 * GM * MU / r**3 + 3.0 * L**2 / (MU * r**4)


def turning_points(E, L, r_lo=1e-3, r_hi=1.0e3, n_scan=200_000):
    """Radial turning points: the roots of E − V_eff(r) (eq-turning).

    Two roots bound a closed orbit; found by a sign-change scan plus Brent
    refinement.

    Parameters
    ----------
    E : float
        Orbit energy.
    L : float
        Angular momentum.
    r_lo, r_hi : float, optional
        Search bounds.
    n_scan : int, optional
        Scan resolution.

    Returns
    -------
    numpy.ndarray
        The turning-point radii.
    """
    return mechanics.turning_points(E, lambda rr: V_eff(rr, L), r_lo, r_hi, n_scan)


def conic_orbit(L, e):
    """Analytic Kepler conic r(φ) from angular momentum and eccentricity (eq-conic).

    Parameters
    ----------
    L : float
        Angular momentum.
    e : float
        Eccentricity.

    Returns
    -------
    phi, r : numpy.ndarray
        The orbit in polar form.
    """
    p_inv = GM * MU**2 / L**2  # = 1/semi-latus-rectum (reduced units)

    def u_of_phi(phi):
        return p_inv * (1.0 + e * np.cos(phi))

    return u_of_phi


def binet_rhs(_phi, y, L):
    """Inverse-square Binet equation as a first-order system (eq-binet).

    u'' + u = GM·μ^2/L^2 in u = 1/r versus φ; its solution is the conic.

    Parameters
    ----------
    _phi : float
        Angle (unused; solver signature).
    y : array_like
        State ``[u, u']``.
    L : float
        Angular momentum.

    Returns
    -------
    list
        The derivative ``[u', u'']``.
    """
    u, dudphi = y
    return [dudphi, GM * MU**2 / L**2 - u]


def cartesian_rhs(_t, s):
    """2-D Newtonian central-force state derivative (reused from §1.4)."""
    x, y, vx, vy = s
    r3 = (x * x + y * y) ** 1.5
    return [vx, vy, -GM * MU * x / r3, -GM * MU * y / r3]

## Exercise 1 — The effective potential

Everything in this chapter is read off one curve. The effective potential
{eq}`eq-veff` superposes the attractive well $V(r)=-GM\mu/r$ (which pulls inward)
on the repulsive **centrifugal barrier** $L^2/(2\mu r^2)$ (which the conserved
angular momentum {eq}`eq-L` erects near the origin). Their sum has a well with a
single minimum, and *that* shape dictates every orbit. Larger $L$ raises a higher
barrier and pushes the minimum outward.

1. Plot $V_{\mathrm{eff}}(r)$ from {eq}`eq-veff` for several angular momenta
   $L$, on a common axis, marking the attractive well and the centrifugal
   barrier (the `V_eff` helper).
2. Confirm analytically that the well's minimum sits at the circular radius
   $r_c=L^2$ (reduced units) by checking $dV_{\mathrm{eff}}/dr=0$ there
   (`dVeff_dr`), for each $L$. The result is shown in
   {numref}`fig-central-veff`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1 — the circular orbit sits at the $V_{\mathrm{eff}}$ minimum

The minimum of $V_{\mathrm{eff}}$ is where the radial force {eq}`eq-rc` vanishes.
In reduced units that predicts $r_c = L^2$ for every $L$, so $dV_{\mathrm{eff}}/dr$
evaluated at $r_c=L^2$ must be zero.

In [ ]:
for L_val in [0.8, 1.2, 1.6]:
    validate.close(
        dVeff_dr(L_val**2, L_val),
        0.0,
        f"circular orbit sits at the V_eff minimum r_c=L²={L_val**2:.2f}  (L={L_val})",
        atol=1e-6,
    )

## Exercise 2 — Circular orbits and their stability

A minimum of $V_{\mathrm{eff}}$ is a circular orbit; whether it is *stable* depends
on the curvature there. A small radial nudge sees the local quadratic
$V_{\mathrm{eff}}\approx V_{\mathrm{eff}}(r_c)+\tfrac12 V_{\mathrm{eff}}''(r_c)(r-r_c)^2$,
so the orbit oscillates back (stable) when $V_{\mathrm{eff}}''(r_c)>0$ and runs
away (unstable) when it is negative. The inverse-square law gives a genuine well,
so its circular orbits are stable: the reason planets keep their orbits.

1. Take $L=1.2$, locate the circular radius $r_c=L^2$ from {eq}`eq-rc`, and
   evaluate the curvature $V_{\mathrm{eff}}''(r_c)$ (`d2Veff_dr2`).
2. Plot $V_{\mathrm{eff}}(r)$ with the circular orbit marked at the bottom of
   the well and the local parabola overlaid, as in
   {numref}`fig-central-stability`.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — the inverse-square circular orbit is stable

Stability is the single inequality $V_{\mathrm{eff}}''(r_c)>0$: the well curves
upward, so the orbit is a stable equilibrium of the radial 1-D motion.

In [ ]:
validate.check(
    d2Veff_dr2(rc, L_val) > 0,
    "the inverse-square circular orbit is stable (V_eff''(r_c) > 0)",
    f"V_eff''(r_c) = {d2Veff_dr2(rc, L_val):.4f}",
)

## Exercise 3 — Turning points and the radial 1-D picture

For a bound orbit ($E<0$) the radial coordinate is trapped in the well and bounces
between two turning points (the **perihelion** $r_{\min}$ and **aphelion**
$r_{\max}$), exactly where the horizontal energy line $E$ cuts the
$V_{\mathrm{eff}}$ curve, $\dot r=0$ in {eq}`eq-turning`. This is the radial motion
read straight off the 1-D energy diagram. The conic solution {eq}`eq-conic`
predicts these turning points analytically as $a(1\mp e)$ with semi-major axis
$a=-GM\mu/(2E)$: a clean cross-check.

1. Take $L=1$ and $E=-0.32$ (a bound orbit). Find $r_{\min},r_{\max}$ as the
   roots of {eq}`eq-turning` with `turning_points` (a sign-change scan refined
   by `scipy.optimize.brentq`).
2. Plot $V_{\mathrm{eff}}(r)$ with the energy line $E$ and the two turning
   points marked ({numref}`fig-central-turning`), shading the classically
   allowed region $E\geq V_{\mathrm{eff}}$.

In [ ]:
# (solution hidden on the public site)


### Validation 3 — turning points equal the conic's $a(1\mp e)$

The turning points found numerically from $E=V_{\mathrm{eff}}$ must coincide with
the perihelion and aphelion of the analytic conic {eq}`eq-conic`, $a(1-e)$ and
$a(1+e)$: two independent routes to the same two radii.

In [ ]:
validate.close(
    [r_min, r_max],
    [a_orb * (1 - e_orb), a_orb * (1 + e_orb)],
    "turning points equal a(1∓e) from the conic",
    rtol=1e-6,
)

## Exercise 4 — The Binet equation reproduces the conic (worked)

The energy diagram gives the radial *range* but not the orbit's *shape*. For that
we switch from time to angle: with $u=1/r$, the radial equation becomes the linear
Binet oscillator {eq}`eq-binet`, whose solution is the conic {eq}`eq-conic`. Rather
than quote that solution we **integrate the Binet ODE numerically** from
perihelion and check that it traces the analytic conic: a derivation made
concrete.

1. Use the same orbit ($L=1$, $E=-0.32$, hence $e\approx0.6$). Start at
   perihelion: $u(0)=(1+e)/p$ with $p=L^2/(GM\mu^2)$, and $u'(0)=0$. Integrate
   {eq}`eq-binet` over $\varphi\in[0,2\pi]$ with `scipy.integrate.solve_ivp`
   (`DOP853`, `rtol=1e-11`, `atol=1e-13`) on a dense `phi`-grid.
2. Overlay the integrated $u(\varphi)$ on the analytic conic $u(\varphi)$
   from {eq}`eq-conic`, and plot the corresponding orbit $r(\varphi)$ in the
   plane ({numref}`fig-central-binet`).

In [ ]:
# (solution hidden on the public site)


### Validation 4 — the Binet integration is the conic

The numerically integrated Binet solution must match the closed-form conic
{eq}`eq-conic` pointwise: same orbit, one route analytic and one numerical.

In [ ]:
validate.close(
    u_binet,
    u_conic,
    "the Binet equation yields the conic-section orbit",
    rtol=1e-6,
)

## Exercise 5 — Apsidal angle and orbit closure (worked animation)

Why does the Kepler ellipse *close*? Because the Binet oscillator {eq}`eq-binet`
advances $u$ through exactly one full cycle as $\varphi$ goes from perihelion to
perihelion: the angle from perihelion to aphelion (the **apsidal angle**) is
exactly $\pi$, so after $2\pi$ the orbit returns to its start. The apsidal line
(the major axis joining perihelion and aphelion through the focus) never rotates.
This is the analytic statement of what [§1.4](../01-elementary-mechanics/kepler-orbits.ipynb) found by integration. The apsidal
geometry is sketched in {numref}`fig-central-apsides`; the animation traces the
closed ellipse, and the validation measures the apsidal angle of the *animated
orbit*, not the animation object.

This is the worked animation; you build the second in Exercise 6.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — the apsidal angle is 180° (the orbit closes)

The defining property of the inverse-square orbit: the angle swept from perihelion
to aphelion is $180^\circ$. Measured from the animated orbit data, it must equal
$180^\circ$ to within the angular grid resolution: the orbit closes.

In [ ]:
validate.close(
    apsidal_angle_deg,
    180.0,
    "inverse-square apsidal angle is 180° (closed orbit)",
    atol=0.5,
)

## Exercise 6 — Precession under a modified force law (student-implemented animation)

The closure of the orbit is a knife-edge property of the $1/r^2$ law. Add even a
small extra term and the apsidal angle stops being $180^\circ$: the ellipse fails
to close, and instead its apsidal line slowly **precesses**, tracing a rosette.
We model this with a relativistic-like correction to the Binet equation,

$$\frac{d^2u}{d\varphi^2} + u = \frac{GM\mu^2}{L^2} + \beta\,u^2,$$

the same $u^2$ term that produces Mercury's perihelion advance in general
relativity (a forward link to "Outlook"). This is the **analytical sibling
of Exercise 7 of [§1.4](../01-elementary-mechanics/kepler-orbits.ipynb)**, which integrated a modified force law in Cartesian
coordinates and watched the same rosette appear.

This is the **student-implemented** animation: the modified dynamics is given;
you build the player and measure the precession.

1. Integrate the modified Binet equation above with
   `scipy.integrate.solve_ivp` (`DOP853`) over several orbits (use
   $\beta=0.1$, $L=1$, $e=0.6$ start), storing $u(\varphi)$.
2. **Build the animation** of the precessing rosette $r(\varphi)=1/u$ in the
   plane, with the slowly rotating apsidal line (there are many valid ways to
   draw it), then `plt.close(fig)` and end with `ecp.animate.show(anim)`.
3. Detect successive perihelia (interior maxima of $u$ on the dense grid) and
   measure the **precession per orbit**: the perihelion-to-perihelion angle
   minus $360^\circ$.

> A ✗ on the final check is about the **perihelion detection or the orbit data**,
> not the drawing: any correct animation of the same data is fine. If it fails,
> check that consecutive $u$-maxima are being found on a finely sampled grid.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — the modified force law precesses

The signature of any non-inverse-square central force: the orbit does not close,
so the precession per orbit is nonzero. We require a clear, measurable advance
(well above a degree) from the animated orbit's perihelia.

In [ ]:
validate.check(
    abs(precession_per_orbit_deg) > 1.0,
    "the modified force law precesses the orbit",
    f"precession = {precession_per_orbit_deg:+.2f}°/orbit",
)

## Exercise 7 — Cross-check: full 2-D integration conserves $E$ and $L$

Our whole reduction rested on two conserved quantities. As an independent close,
we drop back to the *unreduced* picture and integrate the orbit in Cartesian
coordinates with the Newtonian force (the right-hand side from [§1.4](../01-elementary-mechanics/kepler-orbits.ipynb)),
making no use of $V_{\mathrm{eff}}$ or Binet. If the analysis is right, the
Cartesian orbit must (i) conserve the energy $E$ and angular momentum $L$ that
{eq}`eq-veff` and {eq}`eq-L` assumed constant, and (ii) trace the same conic
{eq}`eq-conic` we derived.

1. Launch the same orbit from perihelion ($x=r_{\min}$, $v_y=L/r_{\min}$)
   and integrate `cartesian_rhs` with `scipy.integrate.solve_ivp` (`DOP853`)
   over several periods.
2. Track $E(t)=\tfrac12 v^2 - GM\mu/r$ and $L(t)=x v_y - y v_x$ and confirm
   both are conserved; overlay the Cartesian orbit on the analytic conic
   ({numref}`fig-central-2d`).

In [ ]:
# (solution hidden on the public site)


### Validation 7 — energy and angular momentum are conserved

The two constants the reduction assumed must hold along the fully independent 2-D
trajectory: $E$ and $L$ may drift only at the integrator's tolerance.

In [ ]:
validate.conserved(E_series, "energy conserved (2-D integration)", rel_drift=1e-6)
validate.conserved(
    L_series, "angular momentum conserved (2-D integration)", rel_drift=1e-6
)

## Exercise 8 — What makes an orbit precess: an exactly solvable case (worked)

Exercise 6 showed that a modified force law precesses the orbit. We can now make that
*quantitative*, because there is a perturbation the effective potential solves exactly. Add a
small extra inverse-square term to the potential,

$$V(r)=-\frac{k}{r}-\frac{\beta}{r^2},$$

which contributes an inverse-cube force. The point is what it does to the effective potential:
the extra $-\beta/r^2$ merges with the centrifugal term $L^2/2\mu r^2$ into a single
$1/r^2$ piece with coefficient $(L^2/2\mu-\beta)$ {eq}`eq-veff`. The radial problem is
therefore *identical* to Kepler's but with a shifted angular-momentum barrier, so it can be
solved in closed form. The apsidal angle — perihelion to aphelion — becomes

$$\Phi=\frac{\pi}{\sqrt{1-2\beta/L^2}},$$

and the perihelion advances by $2(\Phi-\pi)=2\pi\!\left(1/\sqrt{1-2\beta/L^2}-1\right)$ each
orbit. Because the case is exactly solvable, this is not a small-$\beta$ approximation; it is
the precise rate, and our integrated orbit must match it at any $\beta$
({numref}`fig-central-precess-exact`).

**This exercise (worked).** For a reference orbit ($L=1$, $e=0.4$, so
$a=1/(1-e^2)$) with $\beta=0.05$:

1. Integrate the perturbed motion in Cartesian coordinates with
   `scipy.integrate.solve_ivp` (`DOP853`), locating each perihelion with a
   `solve_ivp` event on $\mathbf r\cdot\mathbf v=0$.
2. Measure the per-orbit advance of the perihelion *direction* (the change in
   $\operatorname{atan2}(y,x)$ at successive perihelia, via `numpy.unwrap`).
3. Confirm it matches the exact apsidal-angle formula.

In [ ]:
# (solution hidden on the public site)


### Validation 8 — precession matches the exact apsidal-angle formula

In [ ]:
validate.close(
    precession_numeric,
    precession_formula,
    "a small departure from 1/r² makes the orbit precess, at the rate the apsidal-angle formula predicts",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — The general-relativistic term, and Mercury's anomaly (student)

The mechanism is general: *any* departure from $1/r^2$ precesses the orbit. Exercise 6 used the
$\beta u^2$ correction to the Binet equation — an inverse-fourth-power force — and this is the
exact form general relativity adds. The capstone of Volume IV ([§4.8](../04-special-relativity/gr-capstone.ipynb)) derives it: GR
modifies the orbit equation by precisely $3GM u^2/c^2$, and the resulting precession is
$6\pi GM/[a(1-e^2)c^2]$ per orbit. Here we confirm that the GR-form term, with its *physical*
coefficient, reproduces that prediction — and then we lay out where Mercury's famous number
actually comes from.

**This exercise (student).**

1. Reuse the modified Binet right-hand side `binet_rhs_mod` from Exercise 6
   with the relativistic coefficient $\beta=3GM/c^2$, in toy units with
   $c=100$ (so the perturbation is small), integrating with
   `scipy.integrate.solve_ivp` (`DOP853`) and a `solve_ivp` event on
   $du/d\varphi=0$ to pin each perihelion.
2. Confirm the measured precession matches $6\pi GM/[a(1-e^2)c^2]$.

In [ ]:
# (solution hidden on the public site)


### Validation 9 — the GR-form term gives the GR precession formula

In [ ]:
validate.check(
    precession_gr > 1e-5,
    "the inverse-fourth (general-relativistic) term precesses the orbit",
)
validate.close(
    precession_gr,
    precession_gr_formula,
    "the GR-form term gives exactly the precession 6πGM/(a(1−e²)c²)",
    rtol=1e-2,
)

### Mercury's perihelion: an honest accounting

We can now state the famous puzzle truthfully, which matters, because it is easy to tell it
wrong. Mercury's perihelion advances by about $574''$ per century relative to the fixed stars.
Where does it come from?

- An **isolated** Sun–Mercury system would not precess **at all**. By the closure of the
  inverse-square orbit ([§1.4](../01-elementary-mechanics/kepler-orbits.ipynb), the conserved Laplace–Runge–Lenz vector), a pure
  two-body Kepler ellipse is perfectly closed. The naive Sun–Mercury model predicts *zero*
  advance.
- About $531''$/century comes from **Newtonian gravitational tugs of the other planets**,
  which perturb Mercury away from a pure $1/r^2$ field — and, by the mechanism of this
  notebook, any such departure makes the orbit precess.
- That leaves a residual of about $43''$/century that **no Newtonian effect explains**. For
  decades this stubborn remainder was the outstanding anomaly of celestial mechanics.

The resolution is the term we just integrated. General relativity adds an inverse-fourth-power
correction to the gravitational force — precisely the $3GMu^2/c^2$ form above — and it supplies
exactly the missing $43''$ per century. The capstone [§4.8](../04-special-relativity/gr-capstone.ipynb) carries out that calculation
with the physical constants and lands on the observed number. The point worth holding onto is
that the reader now has, in the very code of this notebook, everything needed to *see* the
puzzle: an isolated orbit closes, any non-$1/r^2$ term turns it, and a $1/r^4$ term turns it at
a computable rate. General relativity supplies the missing piece, and [§4.8](../04-special-relativity/gr-capstone.ipynb) collects it.

## Notebook summary

- The **effective potential** $V_{\rm eff}(r)=V(r)+L^2/2\mu r^2$ — one line of physics,
  written out — circular orbits and their stability, and
  the turning points that frame the radial 1-D picture.
- The **Binet equation** reproducing the Kepler conic; the apsidal angle and orbit closure;
  precession under a modified force law; and a full 2-D integration conserving both energy
  and angular momentum.
- **What makes an orbit precess.** An exactly solvable $-\beta/r^2$ perturbation shifts the
  centrifugal barrier and advances the perihelion by exactly $2\pi(1/\sqrt{1-2\beta/L^2}-1)$;
  the general-relativistic $3GMu^2/c^2$ term gives $6\pi GM/[a(1-e^2)c^2]$. An isolated Kepler
  orbit ([§1.4](../01-elementary-mechanics/kepler-orbits.ipynb)) does not precess; planetary tugs supply most of Mercury's $574''$/century, and
  the residual $43''$ is the relativistic piece collected in the capstone [§4.8](../04-special-relativity/gr-capstone.ipynb).

## Outlook

- **Scattering states ($E>0$).** An unbound orbit is a hyperbola with a single
  turning point; the angle through which it is deflected, as a function of the
  impact parameter, is the central-force scattering problem: the direct sequel in
  [§2.5](scattering.ipynb).
- **The Laplace–Runge–Lenz vector.** The inverse-square orbit conserves *one more*
  vector quantity, $\mathbf A = \mathbf p\times\mathbf L - GM\mu^2\,\hat{\mathbf
  r}$, which points along the apsidal line and so explains why it does not
  precess: the "hidden" symmetry hinted at in [§2.2](noether.ipynb).
- **Mercury's perihelion.** The $\beta u^2$ term of Exercise 6 is exactly the form
  of the general-relativistic correction; with the physical coefficient
  $3GM/c^2$ it predicts Mercury's observed $43''$ per century.
- **The other closed orbit.** Bertrand's theorem leaves only one more force law
  with closed orbits (the isotropic Hooke oscillator $V\propto r^2$) whose
  ellipse is centred on, not focused at, the force centre.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()